# PyBullet vs Swift (wind + rlvec) — 6 s Trajectory Comparison with Wind

Runs the **same open-loop motor commands** and the **same wind** through:
- Swift `wind` RK4 (dt = 0.01)  and Swift `rlvec` RK4 (dt = 0.01)
- Swift RK4 fine reference (10 x 0.001 s substeps per control step)
- a diagnostic Swift RK4 **with the gyroscopic term** added (Swift wind/rlvec omit it; PyBullet includes it)
- PyBullet at dt = 0.01 and PyBullet at dt = 0.001 (10 substeps)

Command sequence: generated once by a geometric position/attitude controller flying a 6 s
maneuver (hover -> diagonal move -> climb + 45 deg yaw -> return) in the Swift sim with wind,
then **replayed identically** into every engine (no feedback), so trajectory differences are purely physics.

**Wind model:** defined in Cell 4 as `wind_velocity(t)` + linear drag `F = K_DRAG*(v_wind - v_body)`,
held constant over each 0.01 s control step (world frame) in *both* engines. Replace Cell 4 with the
model from `swift_demo_wind_only.py` to match your demo exactly.

ESC / motor lag are excluded (same as the one-step test): commands are propeller speeds applied directly.

## Steps
1. Cell 1 (install, ~5-10 min for PyBullet)
2. Cell 2: upload `swift_physics_headless_wind.py`, `swift_physics_headless_rlvec.py`, `swift_rl_env.py`
3. Run the rest; download the PNGs and `results_trajectory.txt`

In [ ]:
# Cell 1: Install
!pip install pybullet scipy -q
import pybullet, scipy
print('pybullet OK  scipy', scipy.__version__)

In [ ]:
# Cell 2: Upload files
from google.colab import files
print('Upload swift_physics_headless_wind.py, swift_physics_headless_rlvec.py AND swift_rl_env.py')
uploaded = files.upload()
print('Got:', list(uploaded.keys()))

In [ ]:
# Cell 3: Imports, parameters, timing
import sys, importlib, time, numpy as np, matplotlib.pyplot as plt
import pybullet as p, pybullet_data
sys.path.insert(0, '/content'); sys.path.insert(0, '.')

import swift_physics_headless_wind as wind
try:
    import swift_physics_headless_rlvec as rlvec
except Exception as e:
    raise RuntimeError(f'rlvec import failed (needs swift_rl_env.py and its deps): {e}')
importlib.reload(wind); importlib.reload(rlvec)

params = wind.PARAMS_NANOBENCH
mass, J = params['m'], params['J']
J_diag = np.diag(J)
g_W = params['g_W']
hover_omega = wind.hover_omega_from_params(params)

DT    = 0.01          # control / logging step
T_END = 6.0
N     = int(round(T_END / DT))
SUB   = 10            # substeps for the fine runs (dt = 0.001)
t_grid = np.arange(N + 1) * DT
print(f'{N} control steps, dt={DT}, fine dt={DT/SUB}')
print('wind', wind.__version__, '| rlvec', rlvec.__version__)

# Parameter consistency between the two modules
pr = rlvec.PARAMS_NANOBENCH
print('\nParameter consistency (wind vs rlvec):')
for k in ['m','J','J_mp','g_W','r_P','zeta','spin_sign','c_l','c_d','k_mot','Omega_max']:
    a, b = np.asarray(params[k], float), np.asarray(pr[k], float)
    same = a.shape == b.shape and np.allclose(a, b, rtol=1e-9, atol=0)
    print(f'  {k:10s} {"OK" if same else "MISMATCH"}' + ('' if same else f'  wind={a.tolist()} rlvec={b.tolist()}'))

In [ ]:
# Cell 4: WIND MODEL (replace with the demo's model if different)
# World-frame force from relative airspeed, evaluated at the START of each control step
# and held constant across that step in every engine.
K_DRAG = 0.02   # N per (m/s)

def wind_velocity(t):
    return np.array([1.5 + 1.0*np.sin(2*np.pi*0.4*t),
                    -0.8 + 0.7*np.sin(2*np.pi*0.7*t + 1.0),
                           0.3*np.sin(2*np.pi*1.1*t)])

def wind_force(t, v_body):
    return K_DRAG * (wind_velocity(t) - v_body)      # world frame [N]

In [ ]:
# Cell 5: Wrench, Swift stepping helpers
def wrench(Omega):
    f_props, tau_props = wind.propeller_force_torque(Omega, params['c_l'], params['c_d'], params['spin_sign'])
    return wind.aggregate_forces(f_props, tau_props, params['r_P'])     # thrust/torque only, no gravity

def swift_step(step_fn, state, Omega, t, nsub=1):
    """One control step. Wind enters as a world-frame acceleration folded into g_W (exact ZOH)."""
    f_body, tau_body = wrench(Omega)
    F_w = wind_force(t, state['v_WB'])
    p_t = {**params, 'g_W': params['g_W'] + F_w / mass}
    h = DT / nsub
    for _ in range(nsub):
        state = step_fn(state, f_body, tau_body, p_t, h)
    return dict(p_WB=state['p_WB'], q_WB=state['q_WB'], v_WB=state['v_WB'], omega_B=state['omega_B'])

def init_state():
    return dict(p_WB=np.array([0,0,1.]), q_WB=np.array([1,0,0,0.]),
                v_WB=np.zeros(3), omega_B=np.zeros(3))

def quat_angle_deg(q1, q2):
    """Geodesic angle between two unit quaternions, in degrees. Accurate near 0."""
    d = abs(np.clip(np.dot(q1/np.linalg.norm(q1), q2/np.linalg.norm(q2)), -1, 1))
    return np.degrees(2*np.arccos(d))

In [ ]:
# Cell 6: Reference trajectory + controller (used ONLY to generate the command sequence)
WPS = [(0.0,[0,0,1.0],0.0), (1.0,[0,0,1.0],0.0), (2.5,[1.0,0.5,1.0],0.0),
       (4.0,[1.0,0.5,1.5],np.radians(45)), (6.0,[0,0,1.0],0.0)]

def ref(t):
    for (t0,p0,y0),(t1,p1,y1) in zip(WPS[:-1], WPS[1:]):
        if t <= t1 or (t1, p1, y1) == WPS[-1]:
            s = np.clip((t-t0)/(t1-t0), 0, 1); T = t1-t0
            dp = np.array(p1,float)-np.array(p0,float)
            pos = np.array(p0,float) + dp*(3*s**2-2*s**3)
            vel = dp*6*s*(1-s)/T
            acc = dp*6*(1-2*s)/T**2
            return pos, vel, acc, y0+(y1-y0)*(3*s**2-2*s**3)

KP, KD = 16.0, 7.2                 # position loop (wn=4 rad/s, zeta=0.9)
WN, ZETA = 30.0, 0.8               # roll/pitch attitude loop
WN_YAW   = 3.0                     # yaw loop is much slower (yaw torque authority is small)
WN_VEC   = np.array([WN, WN, WN_YAW])
MAX_TILT = np.radians(30)

def vee(M): return np.array([M[2,1], M[0,2], M[1,0]])

# Allocation: [f0..f3] -> [T, tau_x, tau_y, tau_z]
A = np.zeros((4,4))
for i in range(4):
    fp = np.zeros((4,3)); tp = np.zeros((4,3))
    fp[i] = [0,0,1.0]; tp[i] = [0,0,params['spin_sign'][i]*params['c_d']/params['c_l']]
    fb, tb = wind.aggregate_forces(fp, tp, params['r_P'])
    A[:,i] = [fb[2], *tb]
A_inv = np.linalg.inv(A)

def controller(state, t):
    p_r, v_r, a_r, yaw = ref(t)
    R = wind.quat2rotm_manual(state['q_WB'])
    a_des = a_r + KP*(p_r-state['p_WB']) + KD*(v_r-state['v_WB']) - g_W
    lat = np.linalg.norm(a_des[:2]); lim = np.tan(MAX_TILT)*max(a_des[2], 1e-3)
    if lat > lim: a_des[:2] *= lim/lat
    T_vec = mass*a_des
    b3 = T_vec/np.linalg.norm(T_vec)
    b1c = np.array([np.cos(yaw), np.sin(yaw), 0.0])
    b2 = np.cross(b3, b1c); b2 /= np.linalg.norm(b2)
    Rd = np.column_stack([np.cross(b2, b3), b2, b3])
    eR = 0.5*vee(Rd.T@R - R.T@Rd)
    tau = -J_diag*(WN_VEC**2*eR + 2*ZETA*WN_VEC*state['omega_B'])
    T = max(T_vec @ R[:,2], 0.0)
    f = A_inv @ np.array([T, *tau])
    f = np.clip(f, params['c_l']*50**2, params['c_l']*params['Omega_max']**2)
    return np.sqrt(f/params['c_l'])

# Generate the command sequence closed-loop in Swift (wind on), then freeze it
state = init_state(); OMEGA_CMD = np.zeros((N,4)); track = []
for k in range(N):
    t = k*DT
    OMEGA_CMD[k] = controller(state, t)
    state = swift_step(wind.rk4_rigid_body_step, state, OMEGA_CMD[k], t)
    track.append(np.linalg.norm(state['p_WB'] - ref(t+DT)[0]))
print(f'command generation: max tracking error {max(track)*100:.1f} cm, final {track[-1]*100:.1f} cm')
print(f'Omega range [{OMEGA_CMD.min():.0f}, {OMEGA_CMD.max():.0f}] rad/s (max {params["Omega_max"]:.0f})')

In [ ]:
# Cell 7: Open-loop replay in every engine
def run_swift(step_fn, nsub=1):
    s = init_state(); out = dict(p=[s['p_WB']], q=[s['q_WB']], v=[s['v_WB']], w=[s['omega_B']])
    for k in range(N):
        s = swift_step(step_fn, s, OMEGA_CMD[k], k*DT, nsub)
        out['p'].append(s['p_WB']); out['q'].append(s['q_WB']); out['v'].append(s['v_WB']); out['w'].append(s['omega_B'])
    return {k: np.array(v) for k, v in out.items()}

def setup_pybullet(sub):
    """Create a fresh PyBullet DIRECT instance and return (client, body)."""
    cl = p.connect(p.DIRECT)
    p.resetSimulation(physicsClientId=cl)
    p.setGravity(0, 0, -9.81, physicsClientId=cl)
    p.setTimeStep(DT/sub, physicsClientId=cl)
    col = p.createCollisionShape(p.GEOM_SPHERE, radius=0.05, physicsClientId=cl)
    bd = p.createMultiBody(baseMass=mass, baseCollisionShapeIndex=col, basePosition=[0,0,1], physicsClientId=cl)
    p.changeDynamics(bd, -1, localInertiaDiagonal=tuple(J_diag), linearDamping=0.0, angularDamping=0.0, physicsClientId=cl)
    return cl, bd

def pybullet_read(cl, bd):
    pos, q = p.getBasePositionAndOrientation(bd, physicsClientId=cl)
    vel, omW = p.getBaseVelocity(bd, physicsClientId=cl)
    R = np.array(p.getMatrixFromQuaternion(q, physicsClientId=cl)).reshape(3,3)
    return np.array(pos), np.array([q[3],q[0],q[1],q[2]]), np.array(vel), R.T @ np.array(omW)

def run_pybullet(sub):
    cl, bd = setup_pybullet(sub)
    out = dict(p=[], q=[], v=[], w=[])
    def log():
        a, b, c, d = pybullet_read(cl, bd)
        out['p'].append(a); out['q'].append(b); out['v'].append(c); out['w'].append(d)
    log()
    for k in range(N):
        t = k*DT
        pos, _, vel, _ = pybullet_read(cl, bd)
        F_w = wind_force(t, vel)                   # ZOH over the control step, same as Swift
        f_body, tau_body = wrench(OMEGA_CMD[k])
        for _ in range(sub):
            pos_now = p.getBasePositionAndOrientation(bd, physicsClientId=cl)[0]
            p.applyExternalForce(bd, -1, f_body.tolist(), [0,0,0], p.LINK_FRAME, physicsClientId=cl)
            p.applyExternalForce(bd, -1, F_w.tolist(), list(pos_now), p.WORLD_FRAME, physicsClientId=cl)
            p.applyExternalTorque(bd, -1, tau_body.tolist(), p.LINK_FRAME, physicsClientId=cl)
            p.stepSimulation(physicsClientId=cl)
        log()
    p.disconnect(cl)
    return {k: np.array(v) for k, v in out.items()}

def run_swift_gyro(nsub):
    # DIAGNOSTIC: same RK4 + same wind/commands, but with the gyroscopic term -J^-1 (w x J w) added
    # (PyBullet includes it; the Swift wind/rlvec modules leave it out).
    J_inv = np.linalg.inv(J)
    def deriv(st, f, tau, gw):
        p_, q, v, w = st
        Rm = wind.quat2rotm_manual(q)
        return (v, 0.5*wind.quat_mult(q, np.array([0, *w])), Rm@f/mass + gw, J_inv@(tau - np.cross(w, J@w)))
    s = [np.array([0,0,1.]), np.array([1,0,0,0.]), np.zeros(3), np.zeros(3)]
    out = dict(p=[s[0]], q=[s[1]], v=[s[2]], w=[s[3]]); h = DT/nsub
    for k in range(N):
        f, tau = wrench(OMEGA_CMD[k])
        gw = params['g_W'] + wind_force(k*DT, s[2])/mass
        for _ in range(nsub):
            k1 = deriv(s, f, tau, gw);  s1 = [s[i]+.5*h*k1[i] for i in range(4)]
            k2 = deriv(s1, f, tau, gw); s2 = [s[i]+.5*h*k2[i] for i in range(4)]
            k3 = deriv(s2, f, tau, gw); s3 = [s[i]+h*k3[i] for i in range(4)]
            k4 = deriv(s3, f, tau, gw)
            s = [s[i]+h/6*(k1[i]+2*k2[i]+2*k3[i]+k4[i]) for i in range(4)]; s[1] = s[1]/np.linalg.norm(s[1])
        for key, val in zip(['p','q','v','w'], s): out[key].append(val)
    return {k: np.array(v) for k, v in out.items()}

R = {}
t0 = time.perf_counter(); R['Swift wind  dt=0.01']  = run_swift(wind.rk4_rigid_body_step);  print('swift wind', time.perf_counter()-t0)
t0 = time.perf_counter(); R['Swift rlvec dt=0.01']  = run_swift(rlvec.rk4_rigid_body_step);  print('swift rlvec', time.perf_counter()-t0)
R['Swift wind  dt=0.001'] = run_swift(wind.rk4_rigid_body_step, SUB)
R['PyBullet dt=0.01']     = run_pybullet(1)
R['PyBullet dt=0.001']    = run_pybullet(SUB)
R['Swift+gyro dt=0.001 (diag)'] = run_swift_gyro(SUB)
print('runs:', list(R))

In [ ]:
# Cell 8: Deviation metrics
def dev(a, b):
    return dict(pos=np.linalg.norm(a['p']-b['p'], axis=1)*1000,                    # mm
                att=np.array([quat_angle_deg(x, y) for x, y in zip(a['q'], b['q'])]),  # deg
                vel=np.linalg.norm(a['v']-b['v'], axis=1),                         # m/s
                om =np.linalg.norm(a['w']-b['w'], axis=1))                         # rad/s

S, S_f, P, P_f = R['Swift wind  dt=0.01'], R['Swift wind  dt=0.001'], R['PyBullet dt=0.01'], R['PyBullet dt=0.001']
S_g = R['Swift+gyro dt=0.001 (diag)']
PAIRS = {
    'PyBullet 0.01  vs Swift 0.01  (headline)':               (P,   S),
    'PyBullet 0.001 vs Swift 0.001 (model diff, e.g. gyro)':  (P_f, S_f),
    'Swift 0.01 vs Swift 0.001     (Swift integration err)':  (S,   S_f),
    'PyBullet 0.01 vs PyBullet 0.001 (PyBullet integ. err)':  (P,   P_f),
    'PyBullet 0.001 vs Swift+gyro 0.001 (diagnostic)':        (P_f, S_g),
}
D = {name: dev(a, b) for name, (a, b) in PAIRS.items()}
cross = dev(R['Swift wind  dt=0.01'], R['Swift rlvec dt=0.01'])

L = ['6 s wind trajectory comparison: Swift (wind/rlvec) vs PyBullet',
     f'dt={DT}s, fine dt={DT/SUB}s, K_DRAG={K_DRAG}, open-loop replay of identical Omega commands', '']
for name, d in D.items():
    L.append(name)
    L.append(f'  {"t(s)":>5} {"pos(mm)":>10} {"att(deg)":>10} {"vel(m/s)":>10} {"om(rad/s)":>10}')
    for ts in [1, 2, 3, 4, 5, 6]:
        i = int(round(ts/DT))
        L.append(f'  {ts:>5} {d["pos"][i]:>10.4f} {d["att"][i]:>10.4f} {d["vel"][i]:>10.5f} {d["om"][i]:>10.5f}')
    L.append(f'  max   {d["pos"].max():>10.4f} {d["att"].max():>10.4f} {d["vel"].max():>10.5f} {d["om"].max():>10.5f}')
    L.append('')
L.append(f'wind vs rlvec (dt=0.01) max abs diff: pos {cross["pos"].max()/1000:.2e} m, att {cross["att"].max():.2e} deg, '
         f'vel {cross["vel"].max():.2e}, om {cross["om"].max():.2e}')
summary = '\n'.join(L); print(summary)
open('results_trajectory.txt', 'w').write(summary)

In [ ]:
# Cell 9: Plots
cols = {'Swift wind  dt=0.01':'#3498db', 'Swift rlvec dt=0.01':'#e67e22', 'Swift wind  dt=0.001':'#1abc9c',
        'PyBullet dt=0.01':'#e74c3c', 'PyBullet dt=0.001':'#9b59b6', 'Swift+gyro dt=0.001 (diag)':'#7f8c8d'}
sty  = {'Swift wind  dt=0.01':'-', 'Swift rlvec dt=0.01':':', 'Swift wind  dt=0.001':'--',
        'PyBullet dt=0.01':'-', 'PyBullet dt=0.001':'--', 'Swift+gyro dt=0.001 (diag)':'.'}
ref_pos = np.array([ref(t)[0] for t in t_grid])

# --- Figure 1: trajectories
fig, ax = plt.subplots(1, 4, figsize=(22, 4.8))
for name, r in R.items():
    kw = dict(color=cols[name], ls=sty[name], lw=1.6, label=name)
    ax[0].plot(r['p'][:,0], r['p'][:,1], **kw); ax[1].plot(r['p'][:,0], r['p'][:,2], **kw)
    ax[2].plot(t_grid, r['p'][:,2], **kw)
    ax[3].plot(t_grid, np.degrees([2*np.arctan2(np.linalg.norm(q[1:3]), q[0]) for q in r['q']]), **kw)
ax[0].plot(ref_pos[:,0], ref_pos[:,1], 'k:', lw=1, label='reference')
ax[1].plot(ref_pos[:,0], ref_pos[:,2], 'k:', lw=1)
ax[0].set_xlabel('x (m)'); ax[0].set_ylabel('y (m)'); ax[0].set_title('Top view'); ax[0].axis('equal')
ax[1].set_xlabel('x (m)'); ax[1].set_ylabel('z (m)'); ax[1].set_title('Side view')
ax[2].set_xlabel('t (s)'); ax[2].set_ylabel('z (m)'); ax[2].set_title('Altitude')
ax[3].set_xlabel('t (s)'); ax[3].set_ylabel('tilt (deg)'); ax[3].set_title('Roll/pitch tilt magnitude')
ax[0].legend(fontsize=8); fig.suptitle('Trajectories (identical open-loop commands + wind)', fontweight='bold')
fig.tight_layout(); fig.savefig('trajectories.png', dpi=150, bbox_inches='tight')

# --- Figure 2: deviation vs time
dcol = ['#e74c3c', '#9b59b6', '#3498db', '#2ecc71', '#7f8c8d']
fig, ax = plt.subplots(1, 4, figsize=(22, 4.8))
for a_, key, yl in zip(ax, ['pos','att','vel','om'],
        ['Position deviation (mm)','Attitude deviation (deg)','Velocity deviation (m/s)','Ang.vel deviation (rad/s)']):
    for (name, d), c in zip(D.items(), dcol):
        a_.semilogy(t_grid[1:], np.maximum(d[key][1:], 1e-12), color=c, lw=2 if 'headline' in name else 1.4, label=name)
    a_.set_xlabel('t (s)'); a_.set_ylabel(yl); a_.set_title(yl); a_.grid(alpha=0.3, which='both')
ax[0].legend(fontsize=7.5, loc='lower right'); fig.suptitle('Deviation vs time (log scale)', fontweight='bold')
fig.tight_layout(); fig.savefig('deviation_vs_time.png', dpi=150, bbox_inches='tight')

# --- Figure 3: inputs
wv = np.array([wind_velocity(t) for t in t_grid[:-1]])
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for i in range(4): ax[0].plot(t_grid[:-1], OMEGA_CMD[:,i], label=f'motor {i}')
ax[0].set_xlabel('t (s)'); ax[0].set_ylabel('Omega (rad/s)'); ax[0].set_title('Motor commands (replayed)'); ax[0].legend()
for i, n in enumerate('xyz'): ax[1].plot(t_grid[:-1], wv[:,i], label=f'wind {n}')
ax[1].set_xlabel('t (s)'); ax[1].set_ylabel('m/s'); ax[1].set_title('Wind velocity'); ax[1].legend()
fig.tight_layout(); fig.savefig('inputs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10: Download
from google.colab import files
for f in ['deviation_vs_time.png','trajectories.png','inputs.png','results_trajectory.txt']:
    files.download(f)